# TWZRD Agent Intel — Remote MCP Server with LlamaIndex

This notebook demonstrates using **[TWZRD Agent Intel](https://intel.twzrd.xyz)** — a production remote Streamable-HTTP MCP server — with LlamaIndex agents.

TWZRD Agent Intel trust-scores Solana wallets before agents make HTTP 402 micropayments. No local server setup required — the MCP endpoint is live at `https://intel.twzrd.xyz/mcp`.

## Tools available

| Tool | Cost | Description |
|------|------|-------------|
| `score_agent` | Free | On-chain trust score (0–100) for a Solana wallet |
| `preflight_check` | Free | Boolean readiness — is this wallet safe to pay? |
| `get_trust_receipt` | Paid (x402) | Signed `twzrd.receipt.v5` via USDC micropayment |

## Installation

```bash
pip install llama-index-tools-mcp llama-index-llms-openai
export OPENAI_API_KEY=<your_key>
```


In [ ]:
# Install dependencies
%pip install llama-index-tools-mcp llama-index-llms-openai -q

## Setup — connect to TWZRD Agent Intel MCP server

The `BasicMCPClient` connects to any Streamable-HTTP MCP endpoint. TWZRD's endpoint requires no authentication for the free tools.

In [ ]:
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

TWZRD_MCP_URL = "https://intel.twzrd.xyz/mcp"

mcp_client = BasicMCPClient(TWZRD_MCP_URL)
mcp_tool_spec = McpToolSpec(client=mcp_client)

# List available tools
tools = await mcp_tool_spec.to_tool_list_async()
for t in tools:
    print(f"  {t.metadata.name}: {t.metadata.description}")

## Example 1: Direct tool call — score a Solana wallet

Call `score_agent` directly to get a trust score for a wallet address.

In [ ]:
# Call score_agent directly
wallet = "4LkEFjwNNg6XRSXqD6UFMB6neLEQPKFSVBRzXQo8kNgf"

score_tool = next(t for t in tools if t.metadata.name == "score_agent")
result = await score_tool.acall(wallet=wallet)
print(f"Trust score for {wallet}:\n{result}")

## Example 2: FunctionAgent — trust-gated payment decision

Build a LlamaIndex `FunctionAgent` that uses TWZRD tools to decide whether to approve a USDC payment.

In [ ]:
import os
from llama_index.llms.openai import OpenAI
from llama_index.core.agent.workflow import FunctionAgent

llm = OpenAI(model="gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])

agent = FunctionAgent(
    name="TrustGateAgent",
    description="An agent that verifies Solana wallet trust before approving payments.",
    tools=tools,
    llm=llm,
    system_prompt=(
        "You are a payment safety agent. Before approving any USDC transfer, "
        "you MUST call preflight_check on the recipient wallet. "
        "If preflight passes, also call score_agent and include the trust score in your response. "
        "If preflight fails, decline the payment and explain why."
    ),
)

wallet = "4LkEFjwNNg6XRSXqD6UFMB6neLEQPKFSVBRzXQo8kNgf"
response = await agent.run(
    f"I want to send 5 USDC to {wallet}. Is it safe to proceed?"
)
print(response)

## Example 3: Filter to free tools only

The `get_trust_receipt` tool requires a paid x402 USDC micropayment. For a free-only setup, filter to just the scoring tools.

In [ ]:
from llama_index.tools.mcp import McpToolSpec

# Use only the free tools
free_tool_spec = McpToolSpec(
    client=mcp_client,
    allowed_tools=["score_agent", "preflight_check"],
)
free_tools = await free_tool_spec.to_tool_list_async()
print(f"Free tools loaded: {[t.metadata.name for t in free_tools]}")

agent_free = FunctionAgent(
    name="FreeTrustAgent",
    description="Trust scoring agent using only free TWZRD tools.",
    tools=free_tools,
    llm=llm,
    system_prompt="Check the trust score of Solana wallets before payments.",
)

wallet2 = "DJsfAHjRomMhE7tTfQnFyGKhVVNQHBBXoHF3Rq1uKkVz"
response2 = await agent_free.run(
    f"What is the trust score for wallet {wallet2}?"
)
print(response2)

## MCP server configuration

To use TWZRD Agent Intel in any MCP-compatible client:

```json
{
  "mcpServers": {
    "twzrd-agent-intel": {
      "url": "https://intel.twzrd.xyz/mcp"
    }
  }
}
```

## Resources

- Website: https://intel.twzrd.xyz
- MCP endpoint: https://intel.twzrd.xyz/mcp
- OpenAPI: https://intel.twzrd.xyz/openapi.json
